# Advanced Mutual Fund Analytics

Comprehensive analysis covering:
- VaR/CVaR risk metrics
- Rolling Sharpe ratios
- Investor cohort profiling
- SIP continuity tracking
- Intelligent fund recommendations
- Sector concentration analysis
- Strategic insights for portfolio management

## 1. Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Load cleaned data
nav_df = pd.read_csv('data/processed/nav_history_clean.csv')
scheme_df = pd.read_csv('data/processed/scheme_performance_clean.csv')
transactions_df = pd.read_csv('data/processed/investor_transactions_clean.csv')

# Load raw portfolio holdings for sector analysis
holdings_df = pd.read_csv('data/raw/09_portfolio_holdings.csv')

# Convert date columns
nav_df['date'] = pd.to_datetime(nav_df['date'])
transactions_df['transaction_date'] = pd.to_datetime(transactions_df['transaction_date'])
holdings_df['portfolio_date'] = pd.to_datetime(holdings_df['portfolio_date'])

print("✓ Data loaded successfully")
print(f"  - NAV history: {nav_df.shape[0]:,} records")
print(f"  - Schemes: {scheme_df.shape[0]} funds")
print(f"  - Transactions: {transactions_df.shape[0]:,} records")
print(f"  - Holdings: {holdings_df.shape[0]} holdings")

## 2. Historical VaR and CVaR Analysis

**Value at Risk (VaR)**: 5th percentile of daily return distribution (95% confidence level)
**Conditional Value at Risk (CVaR)**: Mean of returns below the VaR threshold (expected loss in worst 5% cases)

In [ ]:
# Calculate daily returns for all schemes
nav_sorted = nav_df.sort_values(['amfi_code', 'date'])
nav_sorted['daily_return'] = nav_sorted.groupby('amfi_code')['nav'].pct_change() * 100

# Calculate VaR (95%) and CVaR for each scheme
var_cvar_data = []

for amfi_code in scheme_df['amfi_code']:
    returns = nav_sorted[nav_sorted['amfi_code'] == amfi_code]['daily_return'].dropna()
    
    if len(returns) > 0:
        var_95 = np.percentile(returns, 5)  # 5th percentile
        cvar = returns[returns <= var_95].mean()  # Mean of returns <= VaR
        
        # Get scheme name
        scheme_name = scheme_df[scheme_df['amfi_code'] == amfi_code]['scheme_name'].values[0]
        fund_house = scheme_df[scheme_df['amfi_code'] == amfi_code]['fund_house'].values[0]
        category = scheme_df[scheme_df['amfi_code'] == amfi_code]['category'].values[0]
        
        var_cvar_data.append({
            'amfi_code': amfi_code,
            'scheme_name': scheme_name,
            'fund_house': fund_house,
            'category': category,
            'var_95_pct': var_95,
            'cvar_pct': cvar,
            'num_returns': len(returns)
        })

var_cvar_df = pd.DataFrame(var_cvar_data)
var_cvar_df = var_cvar_df.sort_values('var_95_pct', ascending=False)

print("\n📊 Value at Risk & Conditional Value at Risk Summary")
print("="*80)
print(var_cvar_df.head(10).to_string(index=False))
print(f"\n... showing top 10 of {len(var_cvar_df)} schemes")
print(f"\nAverage VaR (95%): {var_cvar_df['var_95_pct'].mean():.4f}%")
print(f"Average CVaR: {var_cvar_df['cvar_pct'].mean():.4f}%")

## 3. Rolling 90-Day Sharpe Ratio Calculation

**Rolling Sharpe Ratio** = (rolling 90-day mean return / rolling 90-day std deviation) × √252
Tracks fund performance stability over time

In [ ]:
# Calculate rolling Sharpe for each scheme
def calculate_rolling_sharpe(returns_series, window=90, periods_per_year=252):
    """Calculate rolling Sharpe ratio"""
    rolling_mean = returns_series.rolling(window=window).mean()
    rolling_std = returns_series.rolling(window=window).std()
    rolling_sharpe = (rolling_mean / rolling_std) * np.sqrt(periods_per_year)
    return rolling_sharpe

# Select top 5 funds by Sharpe ratio for detailed analysis
top_5_funds = scheme_df.nlargest(5, 'sharpe_ratio')[['amfi_code', 'scheme_name']].copy()
print("📈 Rolling Sharpe Ratio - Top 5 Funds Selected:")
print(top_5_funds.to_string(index=False))

# Calculate rolling Sharpe for top 5 funds
fig, axes = plt.subplots(5, 1, figsize=(14, 12))

rolling_sharpe_data = {}

for idx, (_, row) in enumerate(top_5_funds.iterrows()):
    amfi_code = row['amfi_code']
    scheme_name = row['scheme_name']
    
    # Get returns for this fund
    returns = nav_sorted[nav_sorted['amfi_code'] == amfi_code]['daily_return'].dropna()
    
    if len(returns) >= 90:
        # Calculate rolling Sharpe
        rolling_sharpe = calculate_rolling_sharpe(returns, window=90)
        rolling_sharpe_data[scheme_name] = rolling_sharpe.dropna()
        
        # Plot
        ax = axes[idx]
        ax.plot(rolling_sharpe.dropna(), label=f'{scheme_name}', linewidth=2, color='steelblue')
        ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Sharpe = 1.0')
        ax.set_title(f'{scheme_name} - 90-Day Rolling Sharpe Ratio', fontweight='bold')
        ax.set_ylabel('Sharpe Ratio')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rolling_sharpe_chart.png', dpi=300, bbox_inches='tight')
print("\n✓ Rolling Sharpe chart saved as 'rolling_sharpe_chart.png'")
plt.show()

# Summary statistics
print("\n📊 Rolling 90-Day Sharpe Ratio Summary:")
for fund_name, sharpe_series in rolling_sharpe_data.items():
    print(f"  {fund_name}:")
    print(f"    Mean: {sharpe_series.mean():.4f}, Std: {sharpe_series.std():.4f}")
    print(f"    Min: {sharpe_series.min():.4f}, Max: {sharpe_series.max():.4f}")

## 4. Investor Cohort Analysis

Group investors by their first transaction year and analyze investment patterns by cohort

In [ ]:
# Get first transaction year for each investor
first_transaction = transactions_df.groupby('investor_id')['transaction_date'].min().reset_index()
first_transaction['cohort_year'] = first_transaction['transaction_date'].dt.year
first_transaction = first_transaction[['investor_id', 'cohort_year']]

# Merge with transactions
transactions_with_cohort = transactions_df.merge(first_transaction, on='investor_id', how='left')

# Filter for SIP transactions only
sip_transactions = transactions_with_cohort[transactions_with_cohort['transaction_type'] == 'SIP'].copy()

# Cohort analysis
cohort_analysis = sip_transactions.groupby('cohort_year').agg({
    'investor_id': 'nunique',  # Number of unique investors
    'amount_inr': ['mean', 'sum'],  # Avg SIP and total invested
    'amfi_code': 'nunique'  # Number of unique funds
}).round(2)

cohort_analysis.columns = ['num_investors', 'avg_sip_amount', 'total_invested', 'num_funds']

# Find top fund preference per cohort
top_fund_per_cohort = []
for year in sorted(sip_transactions['cohort_year'].unique()):
    year_data = sip_transactions[sip_transactions['cohort_year'] == year]
    top_fund = year_data['amfi_code'].value_counts().head(1)
    if len(top_fund) > 0:
        fund_code = top_fund.index[0]
        fund_name = scheme_df[scheme_df['amfi_code'] == fund_code]['scheme_name'].values
        fund_name = fund_name[0] if len(fund_name) > 0 else 'Unknown'
        top_fund_per_cohort.append({
            'cohort_year': year,
            'top_fund_code': fund_code,
            'top_fund_name': fund_name,
            'investments_in_fund': int(top_fund.values[0])
        })

top_fund_df = pd.DataFrame(top_fund_per_cohort)
cohort_analysis = cohort_analysis.merge(top_fund_df, left_index=True, right_on='cohort_year')

print("\n👥 Investor Cohort Analysis (SIP Investors)")
print("="*120)
print(cohort_analysis.to_string())

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Cohort size
ax1 = axes[0, 0]
ax1.bar(cohort_analysis['cohort_year'], cohort_analysis['num_investors'], color='steelblue')
ax1.set_title('Number of Investors by Cohort', fontweight='bold')
ax1.set_xlabel('Cohort Year')
ax1.set_ylabel('Number of Investors')
ax1.grid(True, alpha=0.3, axis='y')

# Average SIP amount
ax2 = axes[0, 1]
ax2.bar(cohort_analysis['cohort_year'], cohort_analysis['avg_sip_amount'], color='coral')
ax2.set_title('Average SIP Amount by Cohort', fontweight='bold')
ax2.set_xlabel('Cohort Year')
ax2.set_ylabel('Amount (₹)')
ax2.grid(True, alpha=0.3, axis='y')

# Total invested
ax3 = axes[1, 0]
ax3.bar(cohort_analysis['cohort_year'], cohort_analysis['total_invested']/1e6, color='lightgreen')
ax3.set_title('Total Invested by Cohort', fontweight='bold')
ax3.set_xlabel('Cohort Year')
ax3.set_ylabel('Amount (₹ Crores)')
ax3.grid(True, alpha=0.3, axis='y')

# Number of funds
ax4 = axes[1, 1]
ax4.bar(cohort_analysis['cohort_year'], cohort_analysis['num_funds'], color='mediumpurple')
ax4.set_title('Number of Unique Funds by Cohort', fontweight='bold')
ax4.set_xlabel('Cohort Year')
ax4.set_ylabel('Number of Funds')
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. SIP Continuity Analysis

Identify at-risk investors with long gaps (>35 days) between consecutive SIP transactions

In [ ]:
# Filter investors with 6+ SIP transactions
sip_data = transactions_df[transactions_df['transaction_type'] == 'SIP'].copy()
sip_investor_counts = sip_data.groupby('investor_id').size()
active_sip_investors = sip_investor_counts[sip_investor_counts >= 6].index

# Analyze SIP continuity for active investors
sip_active = sip_data[sip_data['investor_id'].isin(active_sip_investors)].copy()
sip_active = sip_active.sort_values(['investor_id', 'transaction_date'])

# Calculate gaps between consecutive SIP dates
continuity_data = []

for investor_id in active_sip_investors:
    investor_sips = sip_active[sip_active['investor_id'] == investor_id].sort_values('transaction_date')
    dates = investor_sips['transaction_date'].values
    
    if len(dates) >= 2:
        # Calculate gaps in days
        gaps = [(pd.Timestamp(dates[i+1]) - pd.Timestamp(dates[i])).days for i in range(len(dates)-1)]
        avg_gap = np.mean(gaps)
        max_gap = np.max(gaps)
        
        # Check if at-risk (gap > 35 days)
        at_risk = 'Yes' if max_gap > 35 else 'No'
        
        continuity_data.append({
            'investor_id': investor_id,
            'num_sips': len(dates),
            'avg_gap_days': avg_gap,
            'max_gap_days': max_gap,
            'at_risk': at_risk,
            'total_invested': investor_sips['amount_inr'].sum()
        })

continuity_df = pd.DataFrame(continuity_data)

# Summary statistics
at_risk_count = (continuity_df['at_risk'] == 'Yes').sum()
at_risk_pct = (at_risk_count / len(continuity_df)) * 100

print(f"\n🔍 SIP Continuity Analysis (Investors with 6+ SIPs)")
print("="*80)
print(f"Total investors with 6+ SIPs: {len(continuity_df)}")
print(f"At-risk investors (gap > 35 days): {at_risk_count} ({at_risk_pct:.1f}%)")
print(f"\nContinuity Metrics:")
print(f"  Average gap between SIPs: {continuity_df['avg_gap_days'].mean():.1f} days")
print(f"  Median gap: {continuity_df['avg_gap_days'].median():.1f} days")
print(f"  Max observed gap: {continuity_df['max_gap_days'].max():.0f} days")

# Show at-risk investors
print(f"\n⚠️  Top 10 At-Risk Investors:")
at_risk_df = continuity_df[continuity_df['at_risk'] == 'Yes'].sort_values('max_gap_days', ascending=False).head(10)
print(at_risk_df[['investor_id', 'num_sips', 'avg_gap_days', 'max_gap_days', 'total_invested']].to_string(index=False))

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gap distribution
ax1 = axes[0, 0]
ax1.hist(continuity_df['avg_gap_days'], bins=30, color='steelblue', edgecolor='black')
ax1.axvline(x=35, color='red', linestyle='--', linewidth=2, label='At-Risk Threshold (35 days)')
ax1.set_title('Distribution of Average SIP Gaps', fontweight='bold')
ax1.set_xlabel('Average Gap (days)')
ax1.set_ylabel('Number of Investors')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# At-risk vs Healthy
ax2 = axes[0, 1]
risk_counts = continuity_df['at_risk'].value_counts()
colors = ['#ff6b6b', '#51cf66']
ax2.pie(risk_counts.values, labels=risk_counts.index, autopct='%1.1f%%', colors=colors, startangle=90)
ax2.set_title('Investor SIP Continuity Status', fontweight='bold')

# Max gap by investor
ax3 = axes[1, 0]
sorted_df = continuity_df.sort_values('max_gap_days', ascending=False).head(15)
ax3.barh(range(len(sorted_df)), sorted_df['max_gap_days'], color='coral')
ax3.set_yticks(range(len(sorted_df)))
ax3.set_yticklabels([f"Inv {x[-4:]}" for x in sorted_df['investor_id']], fontsize=8)
ax3.axvline(x=35, color='red', linestyle='--', alpha=0.5)
ax3.set_title('Top 15 Investors by Maximum SIP Gap', fontweight='bold')
ax3.set_xlabel('Max Gap (days)')
ax3.grid(True, alpha=0.3, axis='x')

# Gap vs Total Invested
ax4 = axes[1, 1]
scatter = ax4.scatter(continuity_df['avg_gap_days'], continuity_df['total_invested'], 
                     c=continuity_df['num_sips'], cmap='viridis', alpha=0.6, s=100)
ax4.axvline(x=35, color='red', linestyle='--', alpha=0.5, label='At-Risk Threshold')
ax4.set_title('SIP Gap vs Total Investment', fontweight='bold')
ax4.set_xlabel('Average Gap (days)')
ax4.set_ylabel('Total Invested (₹)')
ax4.legend()
cbar = plt.colorbar(scatter, ax=ax4)
cbar.set_label('Number of SIPs')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Fund Recommender System

Intelligent fund recommendations based on risk appetite and Sharpe ratio performance

In [ ]:
# Fund Recommender Function
def get_fund_recommendations(risk_appetite):
    """
    Generate fund recommendations based on risk appetite.
    
    Parameters:
    risk_appetite (str): 'Low', 'Moderate', or 'High'
    
    Returns:
    DataFrame: Top 3 funds by Sharpe ratio for the risk grade
    """
    valid_appetites = {'Low': 'Low', 'Moderate': 'Moderate', 'High': 'High'}
    
    if risk_appetite not in valid_appetites:
        return f"Invalid risk appetite. Choose from: {list(valid_appetites.keys())}"
    
    # Filter funds by risk grade
    filtered_funds = scheme_df[scheme_df['risk_grade'] == valid_appetites[risk_appetite]].copy()
    
    if len(filtered_funds) == 0:
        return f"No funds found for risk grade: {risk_appetite}"
    
    # Sort by Sharpe ratio descending
    recommended = filtered_funds.nlargest(3, 'sharpe_ratio')[
        ['amfi_code', 'scheme_name', 'fund_house', 'category', 'sharpe_ratio', 'alpha', 'beta', 'std_dev_ann_pct']
    ]
    
    return recommended

# Test recommendations for all risk levels
print("\n💡 Fund Recommendation Engine")
print("="*120)

for risk_level in ['Low', 'Moderate', 'High']:
    print(f"\n🎯 Risk Appetite: {risk_level}")
    print("-"*120)
    recommendations = get_fund_recommendations(risk_level)
    print(recommendations.to_string(index=False))
    print()

# Create visual comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, risk_level in enumerate(['Low', 'Moderate', 'High']):
    recommendations = get_fund_recommendations(risk_level)
    
    ax = axes[idx]
    ax.barh(range(len(recommendations)), recommendations['sharpe_ratio'], color=['#1f77b4', '#ff7f0e', '#2ca02c'])
    ax.set_yticks(range(len(recommendations)))
    ax.set_yticklabels([name[:30] + '...' if len(name) > 30 else name for name in recommendations['scheme_name']], fontsize=9)
    ax.set_xlabel('Sharpe Ratio', fontweight='bold')
    ax.set_title(f'{risk_level} Risk Appetite - Top 3 Funds', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    
    # Add values on bars
    for i, v in enumerate(recommendations['sharpe_ratio']):
        ax.text(v + 0.02, i, f'{v:.2f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Fund recommendation engine ready for deployment")

## 7. Sector HHI Concentration Analysis

**Herfindahl-Hirschman Index (HHI)** = Σ(weight_i²) 
Measures portfolio concentration: HHI > 2500 = Highly Concentrated, HHI < 1500 = Diversified

In [ ]:
# Calculate HHI for each fund (latest portfolio date)
latest_date = holdings_df['portfolio_date'].max()
latest_holdings = holdings_df[holdings_df['portfolio_date'] == latest_date].copy()

hhi_data = []

for amfi_code in latest_holdings['amfi_code'].unique():
    fund_holdings = latest_holdings[latest_holdings['amfi_code'] == amfi_code]
    
    # Get scheme info
    scheme_info = scheme_df[scheme_df['amfi_code'] == amfi_code]
    if len(scheme_info) == 0:
        continue
    
    scheme_name = scheme_info['scheme_name'].values[0]
    category = scheme_info['category'].values[0]
    
    # Calculate HHI = sum of squared weights
    weights = fund_holdings['weight_pct'].values
    hhi = np.sum(weights ** 2)
    
    # Concentration level
    if hhi > 2500:
        concentration = 'Highly Concentrated'
    elif hhi > 1500:
        concentration = 'Moderately Concentrated'
    else:
        concentration = 'Diversified'
    
    # Top 3 sectors
    top_sectors = fund_holdings.nlargest(3, 'weight_pct')[['sector', 'weight_pct']]
    top_3_sectors = ', '.join([f"{row['sector']} ({row['weight_pct']:.1f}%)" 
                               for _, row in top_sectors.iterrows()])
    
    hhi_data.append({
        'amfi_code': amfi_code,
        'scheme_name': scheme_name,
        'category': category,
        'hhi': hhi,
        'concentration': concentration,
        'num_sectors': fund_holdings['sector'].nunique(),
        'top_3_sectors': top_3_sectors
    })

hhi_df = pd.DataFrame(hhi_data).sort_values('hhi', ascending=False)

print("\n🎯 Sector Concentration (HHI) Analysis")
print("="*130)
print(hhi_df[['scheme_name', 'category', 'hhi', 'concentration', 'num_sectors']].head(15).to_string(index=False))
print(f"\n... showing 15 of {len(hhi_df)} funds")

# Summary statistics
print(f"\n📊 HHI Summary Statistics:")
print(f"  Average HHI: {hhi_df['hhi'].mean():.0f}")
print(f"  Median HHI: {hhi_df['hhi'].median():.0f}")
print(f"  Highest HHI: {hhi_df['hhi'].max():.0f} ({hhi_df.loc[hhi_df['hhi'].idxmax(), 'scheme_name']})")
print(f"  Lowest HHI: {hhi_df['hhi'].min():.0f} ({hhi_df.loc[hhi_df['hhi'].idxmin(), 'scheme_name']})")

# Count by concentration level
conc_counts = hhi_df['concentration'].value_counts()
print(f"\n📈 Portfolio Concentration Distribution:")
for conc_level, count in conc_counts.items():
    pct = (count / len(hhi_df)) * 100
    print(f"  {conc_level}: {count} funds ({pct:.1f}%)")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# HHI distribution
ax1 = axes[0, 0]
ax1.hist(hhi_df['hhi'], bins=20, color='steelblue', edgecolor='black')
ax1.axvline(x=1500, color='orange', linestyle='--', linewidth=2, label='Diversified Threshold')
ax1.axvline(x=2500, color='red', linestyle='--', linewidth=2, label='Concentrated Threshold')
ax1.set_title('Distribution of HHI Scores', fontweight='bold')
ax1.set_xlabel('HHI Score')
ax1.set_ylabel('Number of Funds')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# Concentration pie chart
ax2 = axes[0, 1]
colors = ['#51cf66', '#ffd93d', '#ff6b6b']
ax2.pie(conc_counts.values, labels=conc_counts.index, autopct='%1.1f%%', colors=colors)
ax2.set_title('Funds by Concentration Level', fontweight='bold')

# Top most concentrated
ax3 = axes[1, 0]
top_conc = hhi_df.nlargest(10, 'hhi')
ax3.barh(range(len(top_conc)), top_conc['hhi'], color='coral')
ax3.set_yticks(range(len(top_conc)))
ax3.set_yticklabels([name[:25] + '...' if len(name) > 25 else name for name in top_conc['scheme_name']], fontsize=9)
ax3.axvline(x=2500, color='red', linestyle='--', alpha=0.5)
ax3.set_title('Top 10 Most Concentrated Portfolios', fontweight='bold')
ax3.set_xlabel('HHI Score')
ax3.grid(True, alpha=0.3, axis='x')

# HHI vs Number of Sectors
ax4 = axes[1, 1]
colors_by_conc = {'Diversified': '#51cf66', 'Moderately Concentrated': '#ffd93d', 'Highly Concentrated': '#ff6b6b'}
for conc_level in hhi_df['concentration'].unique():
    data = hhi_df[hhi_df['concentration'] == conc_level]
    ax4.scatter(data['num_sectors'], data['hhi'], label=conc_level, s=100, alpha=0.6,
               color=colors_by_conc.get(conc_level, 'gray'))
ax4.set_title('HHI vs Number of Sectors', fontweight='bold')
ax4.set_xlabel('Number of Sectors')
ax4.set_ylabel('HHI Score')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Advanced Insights and Strategic Recommendations

### Key Findings:

#### 1. 📊 Highest Risk Funds (by VaR)
The following funds present the highest downside risk potential, with worst 5% daily losses exceeding market peers:


In [ ]:
# Insight 1: Highest VaR Funds
print("\n📊 INSIGHT 1: Highest Risk Funds (VaR 95%)")
print("="*100)
top_var_funds = var_cvar_df.nlargest(5, 'var_95_pct')
for idx, row in top_var_funds.iterrows():
    print(f"  • {row['scheme_name']}")
    print(f"    VaR (5th percentile): {row['var_95_pct']:.4f}% | CVaR: {row['cvar_pct']:.4f}%")
    print(f"    Category: {row['category']} | Fund House: {row['fund_house']}")
    print()

print("\n💡 Recommendation: These funds suit aggressive investors with high risk tolerance.")
print("   Conservative portfolios should favor funds with lower VaR scores (<-1.5%).")

# Insight 2: Largest Investor Cohorts
print("\n\n👥 INSIGHT 2: Largest Investor Cohorts by Investment Volume")
print("="*100)
top_cohorts = cohort_analysis.nlargest(5, 'total_invested')
for idx, row in top_cohorts.iterrows():
    print(f"  • Cohort {int(row['cohort_year'])}: {int(row['num_investors'])} investors")
    print(f"    Total Invested: ₹{row['total_invested']:,.0f} | Avg SIP: ₹{row['avg_sip_amount']:,.0f}")
    print(f"    Top Fund: {row['top_fund_name']} ({int(row['investments_in_fund'])} investments)")
    print()

print("\n💡 Recommendation: Focus retention efforts on largest cohorts. Their preferences")
print("   indicate market demand patterns and fund performance acceptance.")

# Insight 3: SIP Continuity Rate
print("\n\n🔄 INSIGHT 3: SIP Continuity & At-Risk Investor Status")
print("="*100)
continuity_rate = ((len(continuity_df) - at_risk_count) / len(continuity_df)) * 100
print(f"  • Total Active SIP Investors (6+ SIPs): {len(continuity_df):,}")
print(f"  • Healthy Continuity Rate: {continuity_rate:.1f}%")
print(f"  • At-Risk Investors (gap > 35 days): {at_risk_count} ({at_risk_pct:.1f}%)")
print(f"  • Median Gap Between SIPs: {continuity_df['avg_gap_days'].median():.1f} days")
print(f"  • Maximum Observed Gap: {continuity_df['max_gap_days'].max():.0f} days")

top_at_risk = continuity_df[continuity_df['at_risk'] == 'Yes'].nlargest(3, 'total_invested')
print(f"\n  Top 3 High-Value At-Risk Investors:")
for investor_id, amount in zip(top_at_risk['investor_id'], top_at_risk['total_invested']):
    print(f"    - {investor_id}: ₹{amount:,.0f}")

print("\n💡 Recommendation: Implement engagement programs for at-risk investors.")
print("   Target those with gaps >35 days with personalized outreach and incentives.")

# Insight 4: Sector Concentration Trends
print("\n\n🎯 INSIGHT 4: Portfolio Concentration & Diversification Trends")
print("="*100)
print(f"  • Highly Concentrated Portfolios (HHI > 2500): {len(hhi_df[hhi_df['hhi'] > 2500])} ({(len(hhi_df[hhi_df['hhi'] > 2500])/len(hhi_df)*100):.1f}%)")
print(f"  • Moderately Concentrated (1500-2500): {len(hhi_df[(hhi_df['hhi'] >= 1500) & (hhi_df['hhi'] <= 2500)])} ({(len(hhi_df[(hhi_df['hhi'] >= 1500) & (hhi_df['hhi'] <= 2500)])/len(hhi_df)*100):.1f}%)")
print(f"  • Diversified Portfolios (HHI < 1500): {len(hhi_df[hhi_df['hhi'] < 1500])} ({(len(hhi_df[hhi_df['hhi'] < 1500])/len(hhi_df)*100):.1f}%)")
print(f"\n  • Average HHI: {hhi_df['hhi'].mean():.0f}")
print(f"  • Portfolio Sectors Range: {hhi_df['num_sectors'].min():.0f} to {hhi_df['num_sectors'].max():.0f} sectors per fund")

most_concentrated = hhi_df.nlargest(1, 'hhi').iloc[0]
most_diversified = hhi_df.nsmallest(1, 'hhi').iloc[0]
print(f"\n  • Most Concentrated: {most_concentrated['scheme_name']} (HHI: {most_concentrated['hhi']:.0f})")
print(f"  • Most Diversified: {most_diversified['scheme_name']} (HHI: {most_diversified['hhi']:.0f})")

print("\n💡 Recommendation: Match fund selection to investor risk profile.")
print("   Conservative investors → Diversified funds (HHI < 1500)")
print("   Aggressive investors → Concentrated bets allowed (HHI > 2500)")

# Insight 5: Fund Recommendation Effectiveness
print("\n\n🎯 INSIGHT 5: Risk-Based Fund Recommendation Performance")
print("="*100)
for risk_level in ['Low', 'Moderate', 'High']:
    recs = get_fund_recommendations(risk_level)
    avg_sharpe = recs['sharpe_ratio'].mean()
    print(f"\n  • {risk_level} Risk Funds (Avg Sharpe: {avg_sharpe:.2f}):")
    for idx, rec in recs.iterrows():
        print(f"    ✓ {rec['scheme_name']} (Sharpe: {rec['sharpe_ratio']:.2f}, Alpha: {rec['alpha']:.2f})")

print("\n💡 Recommendation: Use these fund bundles for automated investor matching.")
print("   Implement KYC-based risk profiling to recommend optimal fund combinations.")
print("   Regular quarterly rebalancing recommended based on Sharpe ratio updates.")

## 9. Export Results and Reports

Save all analysis outputs for reporting, dashboarding, and further analysis

In [ ]:
# Export VaR/CVaR Report
var_cvar_export = var_cvar_df.copy()
var_cvar_export = var_cvar_export.round({'var_95_pct': 4, 'cvar_pct': 4})
var_cvar_export.to_csv('var_cvar_report.csv', index=False)
print("✓ Exported: var_cvar_report.csv ({:,} records)".format(len(var_cvar_export)))

# Export Sector HHI Report
hhi_export = hhi_df.copy()
hhi_export = hhi_export.round({'hhi': 0})
hhi_export.to_csv('sector_hhi_analysis.csv', index=False)
print("✓ Exported: sector_hhi_analysis.csv ({:,} records)".format(len(hhi_export)))

# Export SIP Continuity Report
continuity_export = continuity_df.copy()
continuity_export = continuity_export.round({'avg_gap_days': 1})
continuity_export.to_csv('sip_continuity_analysis.csv', index=False)
print("✓ Exported: sip_continuity_analysis.csv ({:,} records)".format(len(continuity_export)))

# Export Investor Cohort Analysis
cohort_export = cohort_analysis.reset_index()
cohort_export = cohort_export.round({'avg_sip_amount': 2, 'total_invested': 2})
cohort_export.to_csv('investor_cohort_analysis.csv', index=False)
print("✓ Exported: investor_cohort_analysis.csv ({:,} records)".format(len(cohort_export)))

# Export Recommendation Engine (as recommender.py)
recommender_code = '''"""
Fund Recommender System
Purpose: Generate intelligent fund recommendations based on risk appetite
Author: Advanced Analytics Pipeline
"""

import pandas as pd
import numpy as np

class FundRecommender:
    """
    Recommends funds based on investor risk profile and fund performance metrics.
    
    Risk Categories:
    - Low: Conservative, stable funds with lower volatility
    - Moderate: Balanced approach with moderate growth potential
    - High: Aggressive funds with high growth potential
    """
    
    def __init__(self, scheme_data_path='data/processed/scheme_performance_clean.csv'):
        """Initialize recommender with scheme data."""
        self.scheme_df = pd.read_csv(scheme_data_path)
        self.valid_risk_levels = {'Low': 'Low', 'Moderate': 'Moderate', 'High': 'High'}
    
    def get_recommendations(self, risk_appetite, num_recommendations=3):
        """
        Generate fund recommendations for given risk appetite.
        
        Parameters:
        -----------
        risk_appetite : str
            'Low', 'Moderate', or 'High'
        num_recommendations : int
            Number of funds to recommend (default: 3)
        
        Returns:
        --------
        pd.DataFrame
            Top funds ranked by Sharpe ratio with full metrics
        """
        if risk_appetite not in self.valid_risk_levels:
            raise ValueError(f"Invalid risk appetite. Choose from: {list(self.valid_risk_levels.keys())}")
        
        # Filter by risk grade
        filtered_funds = self.scheme_df[
            self.scheme_df['risk_grade'] == self.valid_risk_levels[risk_appetite]
        ]
        
        if len(filtered_funds) == 0:
            raise ValueError(f"No funds found for risk grade: {risk_appetite}")
        
        # Rank by Sharpe ratio
        recommended = filtered_funds.nlargest(
            num_recommendations, 
            'sharpe_ratio'
        )[['scheme_name', 'fund_house', 'category', 'sharpe_ratio', 
           'alpha', 'beta', 'std_dev_ann_pct', 'return_3yr_pct']]
        
        return recommended.reset_index(drop=True)
    
    def get_portfolio_allocation(self, risk_appetite):
        """
        Suggest portfolio allocation weights for risk appetite.
        
        Parameters:
        -----------
        risk_appetite : str
            'Low', 'Moderate', or 'High'
        
        Returns:
        --------
        dict
            Allocation strategy with fund weights
        """
        allocations = {
            'Low': {
                'description': 'Conservative Portfolio - Stability Focus',
                'strategy': '60% Low Risk + 40% Moderate Risk',
                'weight': 0.6
            },
            'Moderate': {
                'description': 'Balanced Portfolio - Growth & Stability',
                'strategy': '50% Moderate Risk + 50% High Risk',
                'weight': 0.5
            },
            'High': {
                'description': 'Aggressive Portfolio - Maximum Growth',
                'strategy': '80% High Risk + 20% Moderate Risk',
                'weight': 0.8
            }
        }
        return allocations.get(risk_appetite, None)
    
    def analyze_fund_performance(self, scheme_name):
        """
        Provide detailed analysis for a specific fund.
        
        Parameters:
        -----------
        scheme_name : str
            Name of the fund to analyze
        
        Returns:
        --------
        pd.Series
            Complete fund metrics and analysis
        """
        fund = self.scheme_df[self.scheme_df['scheme_name'] == scheme_name]
        if len(fund) == 0:
            raise ValueError(f"Fund not found: {scheme_name}")
        return fund.iloc[0]


# Example Usage:
if __name__ == "__main__":
    recommender = FundRecommender()
    
    # Get recommendations for each risk profile
    for risk in ['Low', 'Moderate', 'High']:
        print(f"\\n{risk} Risk Appetite:")
        print("-" * 80)
        recommendations = recommender.get_recommendations(risk, num_recommendations=3)
        print(recommendations.to_string(index=False))
        
        allocation = recommender.get_portfolio_allocation(risk)
        print(f"\\nAllocation: {allocation['strategy']}")
'''

with open('recommender.py', 'w') as f:
    f.write(recommender_code)
print("✓ Exported: recommender.py (Fund recommendation engine)")

print("\n" + "="*100)
print("📊 ANALYSIS COMPLETE - ALL DELIVERABLES READY")
print("="*100)
print("\n✅ Outputs Generated:")
print("   1. Advanced_Analytics.ipynb - Complete analysis notebook (this file)")
print("   2. var_cvar_report.csv - VaR/CVaR metrics for all 40 schemes")
print("   3. sector_hhi_analysis.csv - Sector concentration HHI scores")
print("   4. sip_continuity_analysis.csv - SIP continuation tracking data")
print("   5. investor_cohort_analysis.csv - Cohort-wise investment patterns")
print("   6. rolling_sharpe_chart.png - 90-day rolling Sharpe visualizations")
print("   7. recommender.py - Reusable fund recommendation engine")

print("\n📈 Summary Statistics:")
print(f"   • Schemes Analyzed: {len(var_cvar_df)}")
print(f"   • Active SIP Investors (6+ SIPs): {len(continuity_df):,}")
print(f"   • At-Risk Investors: {at_risk_count} ({at_risk_pct:.1f}%)")
print(f"   • Investor Cohorts: {len(cohort_analysis)}")
print(f"   • Portfolio Holdings Analyzed: {len(hhi_df)}")
print(f"   • Average Portfolio HHI: {hhi_df['hhi'].mean():.0f}")

print("\n🚀 Next Steps:")
print("   1. Import var_cvar_report.csv into Power BI for risk dashboards")
print("   2. Implement recommender.py for automated fund matching")
print("   3. Use sip_continuity_analysis.csv for investor engagement campaigns")
print("   4. Monitor cohort_analysis.csv for market segment trends")
print("   5. Share sector_hhi_analysis.csv with portfolio managers")